### BART
- tranasformer 기반의 모델 
    - encoder : 문장을 이해 부분
    - decoder : 문장을 생성 
- Encoder, Decoder 혼합 모델 
- 번역, 요약, 오타자를 찾아서 새로운 텍스트 구성 
- Encoder : BERT모델의 인코더 방식을 사용하여 문장을 이해 
- Decoder : 출력이 되는 문장은 GPT 방식으로 생성 
- 평가 지표를 확인하는 방법은 n-gram을 이용하여 같은 단어를 사용했는가? -> 로그스케일 수치 값을 출력
- input tokenizer와 output의 tokenizer를 따로 사용
- transformer 모델들은 tokenizer는 sentenceepeice를 사용z

In [146]:
# !pip install lighteval
# !pip install rouge_score evaluate

In [147]:
# 문장 간의 검증 지표를 만들어주는 라이브러리
import evaluate
import numpy as np 
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, \
                        DataCollatorForSeq2Seq, Seq2SeqTrainer, \
                        Seq2SeqTrainingArguments


In [148]:
# 특수 토큰 
# <PAD> : 빈칸 채우기 
# <UNK> : OOV
# <SEP> : 2번째 문장 
# <EOS> : 전체 문장의 끝  : 예측값이 언제 끝나는가? </s>
# model_name = "gogamza/kobart-summarization"
model_name = 'digit82/kobart-summarization'

In [149]:
# tokenizer, model 생성 
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast = True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

c:\Users\ekfla\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ekfla\.cache\huggingface\hub\models--digit82--kobart-summarization. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 262/262 [00:00<00:00, 51240.68it/s]


In [150]:
train_docs = [
    '정부는 중소기업 세제 해택과 R&D 세액 공제를 확대한다고 밝혔다', 
    '해당 기업은 분기 실적에서 매출 성장을 기록했으며 신제품 출시를 예고했다'
]
train_sums = [
    '정부가 중소기업 지원을 확대한다.', 
    '기업이 실적 개선과 신제품 출시를 예고했다.'
]

valid_docs = [
    '교육부가 디지털 교과서 도입을 추친한다고 발표했다'
]
valid_sums = [
    '교육부가 디지털 교과서 도입을 추친한다.'
]

In [151]:
# transformer 모델에서 사용하는 데이터의 형태는 Dataset 
# DatasetDict는 Dataset를 한번에 작업하기 위한 Dict 구조
raw_ds = DatasetDict(
    {
        'train' : Dataset.from_dict(
            {
                'document' : train_docs, 
                'summary' : train_sums
            }
        ), 
        'validation' : Dataset.from_dict(
            {
                'document' : valid_docs, 
                'summary' : valid_sums
            }
        )

    }
)

In [152]:
raw_ds

DatasetDict({
    train: Dataset({
        features: ['document', 'summary'],
        num_rows: 2
    })
    validation: Dataset({
        features: ['document', 'summary'],
        num_rows: 1
    })
})

In [153]:
# 입력 / 출력 문장의 최대 길이를 설정 
max_input_len = 512
max_target_len = 128

In [154]:
# tokenizer 함수 
def token_fn(batch):
    # batch : 배치로 묶인 데이터
    # inputs -> 독립 변수 
    inputs = tokenizer(
        batch['document'], 
        max_length = max_input_len, 
        padding = 'max_length',         # 고정 길이의 벡터를 사용
        trunction = True                # 최대 길이보다 큰 경우 자른다.
    )
    # 출력 데이터 인코딩
    labels = tokenizer(
        batch['summary'], 
        max_length = max_target_len, 
        padding = True, 
        truncation = True
    )

    # padding토큰의 인덱스 값은 일반적으로 0
    # labels의 padding토큰의 인덱스 값을 -100으로 변환 
    # -100으로 변환하는 이유는 -> CrossEntropyLoss()에서 -100은 무시 할수 있는 차원으로 구성 
    # tokenizer의 결과 -> attention_mask(실제 토큰, 패딩 토큰), input_ids(인코딩된 단어들)
    labels_ids = np.array( labels['input_ids'] )
    labels_ids[ labels_ids  == tokenizer.pad_token_id ] = -100
    # labels의 데이터 -> 정답 
    # inputs에 labels 새로운 키를 생성하여 데이터를 대입 
    inputs['labels'] = labels_ids.tolist()

    return inputs

In [155]:
# raw_ds 데이터를 token_fn에 대입 
tokenized_ds = raw_ds.map(
    token_fn, 
    batched=True, 
    remove_columns= ['document', 'summary']
)

Map: 100%|██████████| 2/2 [00:00<00:00, 228.08 examples/s]


Map: 100%|██████████| 1/1 [00:00<00:00, 199.01 examples/s]


In [156]:
tokenized_ds

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1
    })
})

In [157]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer= tokenizer, 
    model = model
)

In [158]:
# 검증 지표 선택 
rouge = evaluate.load('rouge')
# BART모델은 생성된 문장과 정답 문장 사이에 얼마나 많은 단어가 공통으로 등장하였는가? 비율을 계산

# 3개의 연산
# ROUGE-1 : 개별 단어(1-gram)가 얼마나 겹치는가?
# ROUGE-2 : 연속된 2단어(2-gram)가 얼마나가 겹치는가?
# ROUGE-L : 가장 길게 공통으로 이어지는 문자열을 기반으로 측정 

In [159]:
# 생성된 문장과 정답 문장을 일반적인 토큰화 작업이 필요 
from konlpy.tag import Komoran
komoran = Komoran()

In [160]:
# 검증 함수 
def metrics(eval_pred):
    # eval_pred : 예측값, 실젯값
    preds, labels = eval_pred

    # padding token의 인덱스의 값으로 labels의 -100의 값들을 재 변경 
    # decode() 함수를 이용해서 인코딩된 단어들을 다시 단어로 변환 
    labels = np.where(
        labels != -100, labels, tokenizer.pad_token_id
    )

    # 텍스트로 디코딩 작업 
    pred_str = tokenizer.batch_decode(preds, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # 문장에서 좌우의 공백이 존재하는 경우에는 다른 값으로 측정하기 때문에 각 문장 별로 좌우의 공백을 제거 
    pred_str = [doc.strip() for doc in pred_str]
    label_str = [doc.strip() for doc in label_str]

    # ROUGE 계산식 
    result = rouge.compute(
        predictions= pred_str, 
        references= label_str, 
        tokenizer = lambda x : komoran.morphs(x) 
    )

    result = { k : round(v * 100, 2) for k, v in result.items() }

    return result

In [161]:
args = Seq2SeqTrainingArguments(
    output_dir= "./kobart", 
    eval_strategy='epoch', 
    save_strategy='epoch', 
    learning_rate= 5e-05,
    num_train_epochs=5, 
    logging_steps=2, 

    load_best_model_at_end= True, 
    metric_for_best_model= 'rougeL', 
    greater_is_better=True, 

    # generate 설정을 변경 
    # 평가 시 직접 문장을 생성할것인가?
    predict_with_generate=True, 
    # 생성할 문장의 최대 토큰의 길이
    generation_max_length= 64, 
    # 데이터 생성 시 문장 후보의 탐색의 개수 
    generation_num_beams= 4
)

In [162]:
# Trainer 생성 
trainer = Seq2SeqTrainer(
    model = model, 
    args = args, 
    train_dataset = tokenized_ds['train'], 
    eval_dataset= tokenized_ds['validation'], 
    processing_class= tokenizer, 
    data_collator= data_collator, 
    compute_metrics= metrics
)

trainer.train()

c:\Users\ekfla\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,No log,1.428917,57.140000,52.630000,57.140000,57.140000
2,4.692037,1.379048,57.140000,52.630000,57.140000,57.140000
3,4.692037,1.194191,57.140000,52.630000,57.140000,57.140000
4,2.482305,1.081345,57.140000,52.630000,57.140000,57.140000
5,2.482305,1.029158,63.640000,50.000000,63.640000,63.640000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.92it/s]
c:\Users\ekfla\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.12it/s]
c:\Users\ekfla\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.81it/s]
c:\Users\ekfla\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|█

TrainOutput(global_step=5, training_loss=3.141214060783386, metrics={'train_runtime': 20.7957, 'train_samples_per_second': 0.481, 'train_steps_per_second': 0.24, 'total_flos': 3048682291200.0, 'train_loss': 3.141214060783386, 'epoch': 5.0})

In [163]:
test_text = """과학기술정보통신부는 초거대 AI 연구 인프라 지원을 강화한다고 밝혔다. 
스타트업 대상으로 GPU 리소스를 확대 제공할 계획이다."""

inputs = tokenizer(
    test_text, 
    return_tensors = 'pt', 
    truction = True, 
    max_length = max_input_len
)
inputs

{'input_ids': tensor([[22954, 15107, 15509, 14697, 14272,  9031,  9776, 23091, 14541, 19420,
         17364, 14884, 15443, 14253, 14130, 14095, 15050, 13328, 11776, 15237,
         15464,   279,   284, 14420, 11319, 15116, 14779, 21131, 14491, 16247]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]])}

In [164]:

generate_ids = model.generate(
    **inputs, 
    # 출력 토근의 최대 길이 
    max_new_tokens = 64, 
    # 출력 토큰의 최소 길이 
    min_new_tokens = 5, 
    # n개의 후보군 문장을 생성하여 가장 가능성이 높은 문장을 선택 
    num_beams = 4,
    # 샘플링을 사용할 것인가? 빔 서치를 사용하게 되면 False / True는 확률에 따라 랜덤 생성
    do_sample = False, 
    # num_beams 중에 문장이 길어질때 점수를 많이 줄것인가? 깍을 것인가 지정 
    # 기본값은 1 / 
    # 1보다 작게 설정 : 짧게 쓸수록 가산점, 컴펙트한 요약
    # 1보다 크게 설정 : 길게 쓸수록 가산점, 문장을 더 길고 풍부하게 생성
    length_penalty = 0.6, 
    # 반복 방지 -> 단어 배치가 똑같은 단어 뮦음의 개수
    no_repeat_ngram_size = 3, 
    # 토큰 반복 패턴이 나타나는 경우 패널티 적용
    repetition_penalty = 1.1, 
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id
)

[transformers] Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=5) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [165]:
print(tokenizer.decode(generate_ids[0], skip_special_tokens=True))

과학기술정보통신부는 초거대 AI 연구 인프라 지원을 강화한다고 밝혔다.


In [166]:
len(generate_ids[0])

17